# Gán nhãn từ loại tiếng Việt bằng HMM

Notebook tái lập thí nghiệm train/dev của nhóm trên UD Vietnamese-VTB. Các cell chạy demo toy, tải và kiểm checksum hai split, thực hiện grid Bigram/Trigram và hiển thị kết quả dev cùng EDA. Phần đánh giá test được mô tả ở cuối.

Trên Google Colab, dùng CPU. Chạy hai ô mã đầu để mở demo, hoặc chọn **Runtime → Run all** để tái lập toàn bộ thí nghiệm. Ô đầu tự tải mã nguồn công khai và cài thư viện. Trên Jupyter, nếu có bản checkout riêng, có thể đặt biến môi trường `NLP_REPO` trỏ đến thư mục gốc.

In [ ]:
import csv
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys

from IPython.display import Image, Markdown, display

cwd = Path.cwd()
candidates = (cwd, cwd.parent, cwd / "Vietnamese-POS-Tagging-HMM")
if os.getenv("NLP_REPO"):
    candidates = (Path(os.environ["NLP_REPO"]),)
repo = next(
    (path.resolve() for path in candidates if (path / "scripts" / "evaluate_dev.py").is_file()),
    None,
)
if repo is None:
    if os.getenv("NLP_REPO"):
        raise FileNotFoundError(f"Thư mục NLP_REPO thiếu scripts/evaluate_dev.py: {os.environ['NLP_REPO']}")
    repo = cwd / "Vietnamese-POS-Tagging-HMM"
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main",
         "https://github.com/PhamHoagn/Vietnamese-POS-Tagging-HMM.git", str(repo)],
        check=True,
    )
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

data_dir = Path(os.getenv("NLP_DATA_DIR", repo / "data/raw/ud-vtb-fa7130860b65")).resolve()
run_dir = Path(os.getenv("NLP_RUN_DIR", repo / ".venv/project-runs/notebook")).resolve()
predictions = run_dir / "predictions_dev.csv"
env = {**os.environ, "PYTHONUTF8": "1"}

def run(*arguments):
    result = subprocess.run(
        [sys.executable, *map(str, arguments)],
        cwd=repo,
        env=env,
        check=True,
        stdout=subprocess.PIPE,
        text=True,
        encoding="utf-8",
    )
    print(result.stdout, end="")

if os.getenv("NLP_SKIP_INSTALL") != "1":
    run("-m", "pip", "install", "-q", "-r", "requirements.txt")

## Demo gán nhãn tiếng Việt

Chạy ô bên dưới và thử “Tôi thấy cái bàn rất đẹp” hoặc “Chúng tôi đang bàn về dự án”. Chọn nhập câu bình thường để tự động tách từ, hoặc phân cách token bằng dấu `|`. Bảng hiển thị nhãn theo ngữ cảnh cùng dự đoán của B0, Bigram và Trigram. Demo hoạt động trong thời gian phiên notebook còn kết nối.

In [ ]:
from http.server import ThreadingHTTPServer
from threading import Thread

from scripts.demo_web import Demo, handler_for
from scripts.prepare_tv1 import pinned_split

if "pos_demo_server" not in globals():
    train_path, _ = pinned_split(data_dir, "train", download=True)
    dev_path, _ = pinned_split(data_dir, "dev", download=True)
    pos_demo_server = ThreadingHTTPServer(
        ("127.0.0.1", 0), handler_for(Demo(train_path, dev_path))
    )
    Thread(target=pos_demo_server.serve_forever, daemon=True).start()

try:
    from google.colab import output
except ImportError:
    display(Markdown(f"[Mở demo trên máy chạy Jupyter](http://127.0.0.1:{pos_demo_server.server_port}/)"))
else:
    output.serve_kernel_port_as_iframe(pos_demo_server.server_port, height=760)

## Demo trên câu toy

Hai câu train toy đủ để chạy B0, Bigram và Trigram. Bảng in nhãn vàng, dự đoán và trạng thái OOV của câu giữ lại. Dùng [ví dụ giải tay](https://github.com/PhamHoagn/Vietnamese-POS-Tagging-HMM/blob/main/docs/TV2_GIAI_TAY_VITERBI.md) để kiểm tra backpointer và hiểu cách tính điểm đường đi.

In [ ]:
run("-m", "scripts.demo_tv2")

## Dữ liệu và tìm cấu hình trên dev

Hai split được ghim theo cùng một commit corpus và kiểm SHA-256 mỗi lần chạy. HMM học từ train, sau đó chọn cấu hình bằng dev. B0 và hai mô hình cố định α=0,1, cutoff=1 làm mốc so sánh. Bảng dự đoán từng token được giữ local trong thư mục Git bỏ qua.

In [ ]:
run(
    "-m", "scripts.prepare_tv1", "--download", "--data-dir", data_dir,
    "--tables-dir", run_dir / "eda", "--figures-dir", run_dir / "figures",
)
run(
    "-m", "scripts.evaluate_dev", "--data", data_dir, "--output", run_dir / "dev",
    "--predictions", predictions,
)
dev_dir = run_dir / "dev"
selection = json.loads((dev_dir / "selection.json").read_text(encoding="utf-8"))
grid_name = "grid_expanded_dev.csv" if selection["protocol"]["expanded_grid"] else "grid_initial_dev.csv"
run(
    "-m", "scripts.plot_dev_grid", "--grid", dev_dir / grid_name,
    "--output", run_dir / "figures" / "grid_sensitivity.png",
)
run(
    "-m", "scripts.prepare_tv1", "--data-dir", data_dir, "--dev-predictions", predictions,
    "--tables-dir", run_dir / "eda", "--figures-dir", run_dir / "figures",
)

In [ ]:
def read_csv(path):
    with path.open(encoding="utf-8", newline="") as stream:
        return list(csv.DictReader(stream))


def show_metrics(path):
    lines = [
        "| Mô hình | Alpha | Cutoff | Accuracy | Macro-F1 | OOV accuracy | No-PUNCT accuracy |",
        "| --- | ---: | ---: | ---: | ---: | ---: | ---: |",
    ]
    columns = ("token_accuracy", "macro_f1", "oov_accuracy", "accuracy_no_punct")
    for row in read_csv(path):
        scores = [f"{float(row[key]):.2%}" for key in columns]
        values = [row["model"], row["alpha"], row["cutoff"], *scores]
        lines.append("| " + " | ".join(values) + " |")
    display(Markdown("\n".join(lines)))


show_metrics(run_dir / "dev" / "metrics_dev.csv")
figures = ("sentence_lengths.png", "upos_distribution.png", "oov_by_upos.png", "grid_sensitivity.png")
for filename in figures:
    display(Image(filename=run_dir / "figures" / filename, width=900))

## Kết quả Test theo cấu hình đã khóa

Lượt Test chính thức chạy theo [biên bản khóa](https://github.com/PhamHoagn/Vietnamese-POS-Tagging-HMM/blob/main/docs/pre_register_test.md). Cell dưới đây kiểm tra mã băm và hiển thị bảng tổng hợp. Xem [quy trình Test](https://github.com/PhamHoagn/Vietnamese-POS-Tagging-HMM/blob/main/docs/TV3_EVALUATION_RUNBOOK.md#đối-chiếu-kết-quả-test) để đối chiếu chi tiết.

In [ ]:
test_dir = repo / "results" / "tables" / "test"
metadata_path = test_dir / "run_metadata.json"
if metadata_path.is_file():
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    for name in ("metrics_test.csv", "paired_bootstrap_test.csv"):
        digest = hashlib.sha256((test_dir / name).read_bytes()).hexdigest()
        if digest != metadata["summary_sha256"][name]:
            raise ValueError(f"Bảng Test lệch mã băm: {name}")
    show_metrics(test_dir / "metrics_test.csv")
    lines = ["| Chênh lệch | Ước lượng (điểm %) | CI 95% (điểm %) |", "| --- | ---: | ---: |"]
    for row in read_csv(test_dir / "paired_bootstrap_test.csv"):
        delta, lower, upper = (100 * float(row[key]) for key in (
            "delta_token_accuracy", "ci95_lower", "ci95_upper"
        ))
        lines.append(f"| {row['left']} − {row['right']} | {delta:+.3f} | [{lower:+.3f}, {upper:+.3f}] |")
    display(Markdown("\n".join(lines)))
else:
    display(Markdown("Bản checkout này chưa có bảng Test đã khóa."))